# Cantill.ai Spend Intelligence — Phase 1: Data Understanding & Quality Assessment

**Purpose:** inspect the supplied Company A and Company B files to identify data characteristics and risks that affect the reusable pipeline.

This notebook is for development-time analysis only. The production pipeline must perform its cleaning, matching, inclusion and classification automatically on new data.

## Phase 1 outputs
- Dataset inventory and schema checks
- Missing, blank and placeholder-value report
- Exact duplicate and unique-ID checks
- Company B date-quality checks
- Numeric amount-quality analysis
- Company A negative-transaction and duplicate-candidate inspection
- Company B seed-label coverage and exclusion-pattern analysis
- Taxonomy and label coverage checks

**Important:** EDA identifies patterns; it does not automatically define business rules. Do not drop negative amounts or candidate duplicates based only on these reports.


## 0. Imports and project paths
Run this notebook from the project root, where the `data/` directory is located. The expected layout is `data/company_a/` and `data/company_b/`.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 120)

# Find the project root without hardcoding a machine-specific absolute path.
candidate_roots = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "data" / "company_a").exists()
     and (p / "data" / "company_b").exists()),
    Path.cwd()
)

DATA_DIR = PROJECT_ROOT / "data"
COMPANY_A_DIR = DATA_DIR / "company_a"
COMPANY_B_DIR = DATA_DIR / "company_b"

print("Project root:", PROJECT_ROOT)
print("Company A directory exists:", COMPANY_A_DIR.exists())
print("Company B directory exists:", COMPANY_B_DIR.exists())

if not (COMPANY_A_DIR.exists() and COMPANY_B_DIR.exists()):
    raise FileNotFoundError(
        "Could not find data/company_a and data/company_b. "
        "Place the extracted task data under the project root's data/ directory."
    )


Project root: c:\Users\madee\OneDrive\Desktop\cantill\cantill-spend-intelligence
Company A directory exists: True
Company B directory exists: True


## 1. Load all source files
Load mixed-type ERP columns with `low_memory=False`. Keep the raw DataFrames unchanged; create derived columns later for analysis.

In [2]:
erp = pd.read_csv(COMPANY_A_DIR / "erp_goods_receipts.csv", low_memory=False)
invoice_tool = pd.read_csv(COMPANY_A_DIR / "invoice_tool_export.csv", low_memory=False)
labels_a = pd.read_csv(COMPANY_A_DIR / "labels_A.csv", low_memory=False)
taxonomy_a = pd.read_csv(COMPANY_A_DIR / "taxonomy_A.csv", low_memory=False)

invoices_b = pd.read_csv(COMPANY_B_DIR / "invoice_lines.csv", low_memory=False)
fx_rates = pd.read_csv(COMPANY_B_DIR / "fx_rates_usd.csv", low_memory=False)
seed_labels = pd.read_csv(COMPANY_B_DIR / "seed_labels.csv", low_memory=False)
taxonomy_b = pd.read_csv(COMPANY_B_DIR / "taxonomy_B.csv", low_memory=False)

datasets = {
    "A_ERP": erp,
    "A_Invoice_Tool": invoice_tool,
    "A_Golden_Labels": labels_a,
    "A_Taxonomy": taxonomy_a,
    "B_Invoice_Lines": invoices_b,
    "B_FX_Rates": fx_rates,
    "B_Seed_Labels": seed_labels,
    "B_Taxonomy": taxonomy_b,
}

inventory = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    }
    for name, df in datasets.items()
]).set_index("dataset")

display(inventory)


,rows,columns,memory_mb
dataset,,,
A_ERP,37029,26,34.95
A_Invoice_Tool,28520,11,15.77
A_Golden_Labels,37029,6,13.63
A_Taxonomy,40,3,0.01
B_Invoice_Lines,41028,28,49.07
B_FX_Rates,683,3,0.08
B_Seed_Labels,3000,6,0.80
B_Taxonomy,40,3,0.01


## 2. Inspect schemas and required columns
This checks whether the expected fields exist. A missing required field should be handled as a configuration/schema error, not silently ignored.

In [3]:
for name, df in datasets.items():
    print(f"\\n===== {name} ({len(df):,} rows x {df.shape[1]} columns) =====")
    display(pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "null": df.isna().sum()
    }))

required_columns = {
    "A_ERP": ["line_id", "value_zar", "posting_date", "vendor_id", "supplier_name",
              "po_number", "po_item", "material_short_text", "service_short_text",
              "quantity", "spend_type"],
    "A_Invoice_Tool": ["tool_record_id", "invoice_number", "invoice_date",
                       "supplier_name_as_captured", "po_reference", "line_text",
                       "gross_amount", "vat_rate", "currency", "status"],
    "A_Golden_Labels": ["record_id", "category_l1", "category_l2", "category_l3",
                        "supplier_key", "item_key"],
    "A_Taxonomy": ["category_l1", "category_l2", "category_l3"],
    "B_Invoice_Lines": ["line_id", "posting_date", "invoice_date", "due_date",
                        "paid_date", "line_description", "currency", "currency_rate",
                        "line_amount", "vendor_id", "document_type"],
    "B_FX_Rates": ["currency", "month", "units_per_usd"],
    "B_Seed_Labels": ["line_id", "include", "exclusion_reason",
                      "category_l1", "category_l2", "category_l3"],
    "B_Taxonomy": ["category_l1", "category_l2", "category_l3"],
}

schema_rows = []
for name, required in required_columns.items():
    present = set(datasets[name].columns)
    missing = sorted(set(required) - present)
    schema_rows.append({"dataset": name, "required_fields": len(required),
                        "missing_required_fields": missing,
                        "schema_ok": len(missing) == 0})

schema_report = pd.DataFrame(schema_rows)
display(schema_report)
if not schema_report["schema_ok"].all():
    raise ValueError("One or more required columns are missing. Review schema_report.")


\n===== A_ERP (37,029 rows x 26 columns) =====


,dtype,non_null,null
line_id,object,37029,0
source_system,object,37029,0
spend_type,object,37029,0
po_number,int64,37029,0
po_item,float64,37029,0
receipt_document,int64,37029,0
receipt_item,float64,37029,0
posting_date,object,12659,24370
material_number,object,21213,15816
material_short_text,object,37029,0


\n===== A_Invoice_Tool (28,520 rows x 11 columns) =====


,dtype,non_null,null
tool_record_id,object,28520,0
invoice_number,object,28520,0
invoice_date,object,9860,18660
supplier_name_as_captured,object,28520,0
po_reference,object,23321,5199
line_text,object,28520,0
quantity,float64,28520,0
gross_amount,object,28520,0
vat_rate,object,21978,6542
currency,object,28520,0


\n===== A_Golden_Labels (37,029 rows x 6 columns) =====


,dtype,non_null,null
record_id,object,37029,0
category_l1,object,37029,0
category_l2,object,37029,0
category_l3,object,37029,0
supplier_key,object,37029,0
item_key,object,21213,15816


... (7 outputs left out)

,dtype,non_null,null
line_id,object,3000,0
include,int64,3000,0
exclusion_reason,object,450,2550
category_l1,object,2550,450
category_l2,object,2550,450
category_l3,object,2550,450


\n===== B_Taxonomy (40 rows x 3 columns) =====


,dtype,non_null,null
category_l1,object,40,0
category_l2,object,40,0
category_l3,object,40,0


,dataset,required_fields,missing_required_fields,schema_ok
0,A_ERP,11,[],True
1,A_Invoice_Tool,10,[],True
2,A_Golden_Labels,6,[],True
3,A_Taxonomy,3,[],True
4,B_Invoice_Lines,11,[],True
5,B_FX_Rates,3,[],True
6,B_Seed_Labels,6,[],True
7,B_Taxonomy,3,[],True


## 3. Missing, blank and placeholder values
`isna()` alone misses empty strings and text placeholders. The report below counts true nulls, blank strings and common placeholder tokens separately. Placeholder detection is diagnostic; it does not overwrite the raw values.

In [4]:
PLACEHOLDERS = {
    "", " ", "NA", "N/A", "NULL", "NONE", "NAN",
    "NO OBJECT", "PRODUCT", "PARTS", "UNKNOWN", "NOT AVAILABLE"
}

def missing_quality_report(df, dataset_name):
    rows = []
    for col in df.columns:
        s = df[col]
        null_count = int(s.isna().sum())
        text = s.astype("string").str.strip()
        blank_count = int((text.eq("") & s.notna()).sum())
        placeholder_count = int(
            (text.str.upper().isin(PLACEHOLDERS) & s.notna() & ~text.eq("")).sum()
        )
        rows.append({
            "dataset": dataset_name,
            "column": col,
            "null_count": null_count,
            "blank_count": blank_count,
            "placeholder_count": placeholder_count,
            "null_pct": round(null_count / len(df) * 100, 2) if len(df) else 0,
        })
    return pd.DataFrame(rows)

missing_report = pd.concat(
    [missing_quality_report(df, name) for name, df in datasets.items()],
    ignore_index=True
)

display(
    missing_report[
        (missing_report[["null_count", "blank_count", "placeholder_count"]].sum(axis=1) > 0)
    ].sort_values(["dataset", "null_count"], ascending=[True, False])
)


,dataset,column,null_count,blank_count,placeholder_count,null_pct
10,A_ERP,service_short_text,36442,0,0,98.41
22,A_ERP,cost_centre,31756,0,0,85.76
7,A_ERP,posting_date,24370,0,0,65.81
13,A_ERP,uom,24370,0,0,65.81
14,A_ERP,po_uom,24370,0,0,65.81
16,A_ERP,unit_cost_zar,24370,0,0,65.81
18,A_ERP,vendor_id,24370,0,0,65.81
20,A_ERP,site_code,24370,0,0,65.81
11,A_ERP,material_type,19070,0,0,51.50
8,A_ERP,material_number,15816,0,0,42.71


## 4. Exact duplicate rows and identifier uniqueness
Exact row duplicates are different from business duplicates. This section checks exact duplicates and uniqueness of record IDs only.

In [5]:
duplicate_rows = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pct": round(df.duplicated().mean() * 100, 3) if len(df) else 0
    }
    for name, df in datasets.items()
]).set_index("dataset")
display(duplicate_rows)

id_checks = {
    "A_ERP.line_id": (erp, "line_id"),
    "A_Invoice_Tool.tool_record_id": (invoice_tool, "tool_record_id"),
    "B_Invoice_Lines.line_id": (invoices_b, "line_id"),
    "A_Golden_Labels.record_id": (labels_a, "record_id"),
    "B_Seed_Labels.line_id": (seed_labels, "line_id"),
}
id_report = pd.DataFrame([
    {
        "identifier": name,
        "null_ids": int(df[col].isna().sum()),
        "duplicate_ids": int(df[col].duplicated().sum()),
        "unique_ids": int(df[col].nunique(dropna=True))
    }
    for name, (df, col) in id_checks.items()
]).set_index("identifier")
display(id_report)


,rows,exact_duplicate_rows,duplicate_pct
dataset,,,
A_ERP,37029,0,0.0
A_Invoice_Tool,28520,0,0.0
A_Golden_Labels,37029,0,0.0
A_Taxonomy,40,0,0.0
B_Invoice_Lines,41028,0,0.0
B_FX_Rates,683,0,0.0
B_Seed_Labels,3000,0,0.0
B_Taxonomy,40,0,0.0


,null_ids,duplicate_ids,unique_ids
identifier,,,
A_ERP.line_id,0,0,37029
A_Invoice_Tool.tool_record_id,0,0,28520
B_Invoice_Lines.line_id,0,0,41028
A_Golden_Labels.record_id,0,0,37029
B_Seed_Labels.line_id,0,0,3000


## 5. Company B date quality
Parse dates into separate analysis columns so the original raw columns remain unchanged. Missing `paid_date` is not automatically an exclusion: the task says approved **or** paid invoices count, and Company B does not have a simple status column.

In [6]:
B_DATE_COLUMNS = ["posting_date", "invoice_date", "due_date", "paid_date"]
b_dates = pd.DataFrame(index=invoices_b.index)

date_parse_rows = []
for col in B_DATE_COLUMNS:
    parsed = pd.to_datetime(invoices_b[col], errors="coerce")
    b_dates[col] = parsed
    raw_present = invoices_b[col].notna() & invoices_b[col].astype("string").str.strip().ne("")
    date_parse_rows.append({
        "column": col,
        "raw_missing_or_blank": int((~raw_present).sum()),
        "unparseable_nonblank": int((raw_present & parsed.isna()).sum()),
        "parsed_missing_total": int(parsed.isna().sum()),
    })

date_parse_report = pd.DataFrame(date_parse_rows).set_index("column")
display(date_parse_report)

date_checks = {
    "invoice_date_after_posting_date":
        b_dates["invoice_date"] > b_dates["posting_date"],
    "paid_date_before_posting_date":
        b_dates["paid_date"] < b_dates["posting_date"],
    "paid_date_before_invoice_date":
        b_dates["paid_date"] < b_dates["invoice_date"],
    "due_date_before_invoice_date":
        b_dates["due_date"] < b_dates["invoice_date"],
    "due_date_before_posting_date":
        b_dates["due_date"] < b_dates["posting_date"],
    "invoice_over_365_days_before_posting":
        b_dates["invoice_date"] < (b_dates["posting_date"] - pd.Timedelta(days=365)),
}

date_consistency_report = pd.DataFrame([
    {"check": name, "records": int(condition.fillna(False).sum())}
    for name, condition in date_checks.items()
]).set_index("check")
display(date_consistency_report)

# Inspect a bounded sample of inconsistent records; investigate before defining rules.
date_anomaly_mask = pd.concat(
    [condition.fillna(False) for condition in date_checks.values()], axis=1
).any(axis=1)

date_anomaly_columns = [
    "line_id", "source_system", "region", "invoice_number",
    "posting_date", "invoice_date", "due_date", "paid_date",
    "document_type", "line_description", "line_amount", "currency"
]
display(invoices_b.loc[date_anomaly_mask, date_anomaly_columns].head(30))


,raw_missing_or_blank,unparseable_nonblank,parsed_missing_total
column,,,
posting_date,0,0,0
invoice_date,0,0,0
due_date,0,0,0
paid_date,10322,0,10322


,records
check,
invoice_date_after_posting_date,233
paid_date_before_posting_date,34
paid_date_before_invoice_date,16
due_date_before_invoice_date,13
due_date_before_posting_date,4759
invoice_over_365_days_before_posting,147


,line_id,source_system,region,invoice_number,posting_date,invoice_date,due_date,paid_date,document_type,line_description,line_amount,currency
0,B000001,ERP-B1 (JDE),R1,B-INV-1000001,2024-03-29,2018-07-23,2024-03-29,NaN,PD,NaN,-13827.64,USD
1,B000002,ERP-B1 (JDE),R1,B-INV-1000001,2024-03-29,2018-07-23,2024-03-27,NaN,PD,NaN,13827.64,USD
2,B000003,ERP-B1 (JDE),R1,B-INV-1000007,2024-11-11,2023-06-27,2024-11-11,2024-11-05,PD,NaN,-2611.52,USD
3,B000004,ERP-B1 (JDE),R1,B-INV-1000011,2024-01-15,2023-08-01,2023-08-01,2024-01-17,PV,NaN,1507.39,HKD
4,B000005,ERP-B1 (JDE),R1,B-INV-1000014,2024-03-29,2023-09-26,2023-10-26,2024-03-29,PV,NaN,5529.29,EUR
5,B000006,ERP-B1 (JDE),R1,B-INV-1000022,2024-01-30,2023-09-28,2023-10-28,2024-02-05,PV,NaN,30945.60,USD
6,B000007,ERP-B1 (JDE),R1,B-INV-1000025,2024-01-30,2023-11-24,2023-12-24,2024-02-27,PV,"INK,CODER,VISUAL YEL,1 QT - 9 EA = 1 CA",656.02,MMK
7,B000008,ERP-B1 (JDE),R1,B-INV-1000031,2024-01-09,2023-12-02,2023-12-02,2024-01-17,PV,NaN,1507.39,HKD
19,B000020,ERP-B1 (JDE),R1,B-INV-1000059,2024-02-09,2023-12-18,2024-01-17,2024-02-23,PV,NaN,4219.49,USD
20,B000021,ERP-B1 (JDE),R1,B-INV-1000059,2024-02-09,2023-12-18,2024-01-17,2024-02-23,PV,NaN,1211.75,USD


## 6. Company B document types and date-anomaly distribution
Document type is an input feature for investigation, not a status mapping by itself. Do not assume a code such as PV or RE means approved/paid unless the supplied evidence supports it.

In [7]:
display(invoices_b["document_type"].value_counts(dropna=False).rename("records").to_frame())

doc_date_flags = pd.DataFrame({
    "document_type": invoices_b["document_type"],
    **{name: cond.fillna(False) for name, cond in date_checks.items()}
})

display(
    doc_date_flags.groupby("document_type", dropna=False)
    .agg(["size", "sum"])
    .sort_values(("invoice_over_365_days_before_posting", "sum"), ascending=False)
)


,records
document_type,
PV,21138
RE,18824
CA,253
Z6,169
PD,130
Z7,96
PM,88
PL,77
PR,58


invoice_date_after_posting_date       \
                                         size  sum   
document_type                                        
PV                                      21138  232   
RE                                      18824    0   
PD                                        130    1   
PL                                         77    0   
CA                                        253    0   
AB                                         47    0   
KG                                          3    0   
PP                                         34    0   
PM                                         88    0   
PR                                         58    0   
R1                                         11    0   
R3                                          3    0   
Z6                                        169    0   
Z7                                         96    0   
ZE                                         31    0   
ZG                                         46    0   
ZZ                                         20    0   

              paid_date_before_posting_date     paid_date_before_invoice_date  \
                                       size sum                          size   
document_type                                                                   
PV                                    21138  26                         21138   
RE                                    18824   0                         18824   
PD                                      130   1                           130   
PL                                       77   0                            77   
CA                                      253   0                           253   
AB                                       47   0                            47   
KG                                        3   0                             3   
PP                                       34   0                            34   
PM                                       88   0                            88   
PR                                       58   7                            58   
R1                                       11   0                            11   
R3                                        3   0                             3   
Z6                                      169   0                           169   
Z7                                       96   0                            96   
ZE                                       31   0                            31   
ZG                                       46   0                            46   
ZZ                                       20   0                            20   

                  due_date_before_invoice_date      \
              sum                         size sum   
document_type                                        
PV             16                        21138  11   
RE              0                        18824   0   
PD              0                          130   1   
PL              0                           77   0   
CA              0                          253   0   
AB              0                           47   0   
KG              0                            3   0   
PP              0                           34   0   
PM              0                           88   0   
PR              0                           58   0   
R1              0                           11   0   
R3              0                            3   0   
Z6              0                          169   0   
Z7              0                           96   0   
ZE              0                           31   0   
ZG              0                           46   1   
ZZ              0                           20   0   

              due_date_before_posting_date        \
                                      size   sum   
document_type                                      
PV                                   21138  2554   
RE                     

## 7. Numeric amount quality across all amount fields
Convert to numeric in derived Series. This section explicitly handles mixed dtypes and reports values that fail numeric conversion. Do not remove negatives or clip outliers automatically.

In [8]:
amount_sources = {
    "A_ERP_value_zar": erp["value_zar"],
    "A_Invoice_Tool_gross_amount": invoice_tool["gross_amount"],
    "B_line_amount_local_currency": invoices_b["line_amount"],
}

numeric_amounts = {}
amount_summary_rows = []

for name, raw in amount_sources.items():
    numeric = pd.to_numeric(raw, errors="coerce")
    numeric_amounts[name] = numeric
    invalid = raw.notna() & numeric.isna()

    amount_summary_rows.append({
        "dataset_field": name,
        "rows": len(raw),
        "raw_missing": int(raw.isna().sum()),
        "invalid_numeric": int(invalid.sum()),
        "negative": int((numeric < 0).sum()),
        "zero": int((numeric == 0).sum()),
        "positive": int((numeric > 0).sum()),
        "min": numeric.min(),
        "p01": numeric.quantile(0.01),
        "median": numeric.median(),
        "p99": numeric.quantile(0.99),
        "max": numeric.max(),
        "mean": numeric.mean(),
    })

amount_quality_report = pd.DataFrame(amount_summary_rows).set_index("dataset_field")
display(amount_quality_report)

for name, raw in amount_sources.items():
    numeric = numeric_amounts[name]
    invalid = raw.notna() & numeric.isna()
    if invalid.any():
        print(f"Invalid numeric examples: {name}")
        display(raw[invalid].head(10))


,rows,raw_missing,invalid_numeric,negative,zero,positive,min,p01,median,p99,max,mean
dataset_field,,,,,,,,,,,,
A_ERP_value_zar,37029,0,0,434,0,36595,-20503682.72,-3223.3136,16843.56,757735.2000,15967882.19,68951.979434
A_Invoice_Tool_gross_amount,28520,0,7920,231,0,20369,-2223613.25,-927.6843,16717.31,885754.6600,16623893.33,78052.745782
B_line_amount_local_currency,41028,3,0,833,918,39274,-3406645.05,-435.3100,721.15,78211.9728,6924157.27,9103.725939


Invalid numeric examples: A_Invoice_Tool_gross_amount


2        5,266.63
6       27,764.58
13      15,143.41
14    -116,020.33
15      16,520.30
16      74,371.13
20      40,241.88
25      26,628.66
27      18,033.40
28       3,451.01
Name: gross_amount, dtype: object

## 8. Company A negative ERP transactions
Negative spend is not automatically invalid. Inspect transaction context, quantities, PO details and golden labels. The output is evidence for a documented decision, not a deletion instruction.

In [9]:
erp_amount = pd.to_numeric(erp["value_zar"], errors="coerce")
erp_quantity = pd.to_numeric(erp["quantity"], errors="coerce")

negative_erp = erp.loc[erp_amount < 0].copy()
negative_erp["analysis_amount"] = erp_amount.loc[erp_amount < 0]
negative_erp["analysis_quantity"] = erp_quantity.loc[erp_amount < 0]

print("Negative ERP rows:", len(negative_erp))
print("Negative amount total (ZAR):", negative_erp["analysis_amount"].sum())
print("Negative amount with negative quantity:",
      int((negative_erp["analysis_quantity"] < 0).sum()))
print("Negative amount with positive quantity:",
      int((negative_erp["analysis_quantity"] > 0).sum()))
print("Negative amount with zero/missing quantity:",
      int((negative_erp["analysis_quantity"].isna() | (negative_erp["analysis_quantity"] == 0)).sum()))

negative_by_spend_type = (
    negative_erp.assign(analysis_amount=negative_erp["analysis_amount"])
    .groupby("spend_type", dropna=False)
    .agg(
        negative_records=("line_id", "count"),
        total_negative_value=("analysis_amount", "sum"),
        average_negative_value=("analysis_amount", "mean"),
        unique_suppliers=("supplier_name", "nunique"),
    )
    .sort_values("negative_records", ascending=False)
)
display(negative_by_spend_type)

negative_columns = [
    "line_id", "posting_date", "po_number", "po_item",
    "material_short_text", "service_short_text", "quantity",
    "value_zar", "vendor_id", "supplier_name", "spend_type", "opex_capex"
]
display(
    negative_erp.assign(_amount=negative_erp["analysis_amount"])
    .sort_values("_amount")
    [negative_columns].head(25)
)


Negative ERP rows: 434
Negative amount total (ZAR): -56015073.690000005
Negative amount with negative quantity: 433
Negative amount with positive quantity: 1
Negative amount with zero/missing quantity: 0


,negative_records,total_negative_value,average_negative_value,unique_suppliers
spend_type,,,,
Material,280,-48274979.74,-172410.641929,45
Free Text,95,-4287346.40,-45129.962105,25
Catalogue,36,-990162.64,-27504.517778,1
Service,23,-2462584.91,-107068.909130,8


,line_id,posting_date,po_number,po_item,material_short_text,service_short_text,quantity,value_zar,vendor_id,supplier_name,spend_type,opex_capex
5574,A006809,2022-10-19,4510014625,10.0,POWDER;STNEDST,NaN,-33580.00,-20503682.72,V200116,NEXORA (PTY) LTD,Material,Opex
10003,A012252,2023-11-30,4510033733,10.0,"PICK, DRUM CUTTER;P/N:1924847,SHNK:30MM",NaN,-31200.00,-2736115.20,V200041,DUNAX (PTY) LTD,Material,Opex
1770,A002178,2021-11-17,4510006345,10.0,"PICK, DRUM CUTTER;P/N:3113108,SHNK:30MM",NaN,-24100.00,-1933576.74,V200041,DUNAX (PTY) LTD,Material,Opex
28261,A034305,NaN,4510091597,20.0,"BLD,WAL,UG;AIR XNG,SNGL",NaN,-2654.53,-1272386.57,NaN,KINLEY (PTY) LTD,Service,Opex
8154,A010035,2023-06-13,4510027567,20.0,"ROD DRIL;P/N:5950087,CONCTN:M16 DIN 405",NaN,-1570.00,-917625.28,V200041,DUNAX (PTY) LTD,Material,Opex
5811,A007104,2022-11-07,4510019328,20.0,"MESH;4,8M,WD:1,2M,WD 75 X LG 100MM,DIA 4",NaN,-3000.00,-731817.90,V200031,PRATON (PTY) LTD,Material,Opex
28517,A034611,NaN,4510092386,10.0,"PICK, DRUM CUTTER;P/N:5896353,SLIMLN,RND",NaN,-6000.00,-622171.80,NaN,DUNAX (PTY) LTD,Material,Opex
5265,A006435,2022-09-20,4510017742,10.0,"ROD ANCHR;DIA:20MM,2400MM,REB QC(HR),ROF",NaN,-7000.00,-594810.30,V200069,YARAX TECHNOLOGIES (PTY) LTD,Material,Opex
21321,A025893,NaN,4510069407,10.0,"ROD ANCHR;DIA:20MM,1800MM,REB QC(HR),ROF",NaN,-6000.00,-587928.60,NaN,PELFIELD STEEL (PTY) LTD,Material,Opex
21320,A025892,NaN,4510069407,10.0,"ROD ANCHR;DIA:20MM,1800MM,REB QC(HR),ROF",NaN,-6000.00,-587928.60,NaN,PELFIELD STEEL (PTY) LTD,Material,Opex


## 9. Company A golden-label coverage for negative records
Check whether the negative ERP records have category, supplier and item labels. Use the correct `labels_a` variable and validate the one-to-one join.

In [10]:
negative_labelled = negative_erp.merge(
    labels_a,
    left_on="line_id",
    right_on="record_id",
    how="left",
    validate="one_to_one"
)

negative_label_coverage = pd.DataFrame({
    "metric": ["negative records", "missing category L1", "missing supplier_key", "missing item_key"],
    "count": [
        len(negative_labelled),
        int(negative_labelled["category_l1"].isna().sum()),
        int(negative_labelled["supplier_key"].isna().sum()),
        int(negative_labelled["item_key"].isna().sum())
    ]
})
display(negative_label_coverage)

display(
    negative_labelled["category_l1"]
    .value_counts(dropna=False)
    .rename("negative_record_count")
    .to_frame()
)

display(
    negative_labelled[
        ["line_id", "value_zar", "category_l1", "category_l2",
         "category_l3", "supplier_key", "item_key"]
    ].head(20)
)


,metric,count
0,negative records,434
1,missing category L1,434
2,missing supplier_key,434
3,missing item_key,434


,negative_record_count
category_l1,
NaN,434


,line_id,value_zar,category_l1,category_l2,category_l3,supplier_key,item_key
0,A000024,-270831.00,NaN,NaN,NaN,NaN,NaN
1,A000032,-10440.00,NaN,NaN,NaN,NaN,NaN
2,A000033,-20880.00,NaN,NaN,NaN,NaN,NaN
3,A000094,-128064.00,NaN,NaN,NaN,NaN,NaN
4,A000249,-2088.00,NaN,NaN,NaN,NaN,NaN
5,A000253,-2088.00,NaN,NaN,NaN,NaN,NaN
6,A000255,-2088.00,NaN,NaN,NaN,NaN,NaN
7,A000444,-27758.74,NaN,NaN,NaN,NaN,NaN
8,A000581,-27758.74,NaN,NaN,NaN,NaN,NaN
9,A000666,-24840.68,NaN,NaN,NaN,NaN,NaN


## 10. Company A potential internal duplicate candidates
Candidate generation uses PO/item/description/quantity/amount fields. Rows with missing values in signature fields are excluded from this particular signature check to avoid treating many incomplete rows as identical. A repeated signature is **not proof** of a duplicate: multiple receipts can legitimately share a PO/item.

In [11]:
signature_columns = [
    "po_number", "po_item", "material_short_text",
    "service_short_text", "quantity", "value_zar"
]

# Create normalised analysis-only signature values.
erp_signature = erp[signature_columns].copy()
for col in signature_columns:
    erp_signature[col] = erp_signature[col].astype("string").str.strip().str.upper()

complete_signature = erp_signature.notna().all(axis=1)
for col in signature_columns:
    complete_signature &= erp_signature[col].ne("")

candidate_mask = pd.Series(False, index=erp.index)
candidate_mask.loc[complete_signature] = erp_signature.loc[complete_signature].duplicated(
    subset=signature_columns, keep=False
)

possible_duplicates = erp.loc[candidate_mask].copy()
print("Rows with complete signatures:", int(complete_signature.sum()))
print("Potential duplicate candidate rows:", len(possible_duplicates))
print("Distinct repeated signatures:",
      int(erp_signature.loc[candidate_mask].drop_duplicates().shape[0]))

candidate_columns = [
    "line_id", "posting_date", "po_number", "po_item",
    "material_short_text", "service_short_text", "quantity",
    "value_zar", "vendor_id", "supplier_name"
]
display(possible_duplicates[candidate_columns].head(50))


Rows with complete signatures: 587
Potential duplicate candidate rows: 7
Distinct repeated signatures: 3


,line_id,posting_date,po_number,po_item,material_short_text,service_short_text,quantity,value_zar,vendor_id,supplier_name
636,A000775,2021-08-13,4510002381,10.0,"REPR;SUPPORT;ROLLER AND SHAFT,ON :",SER:REPAIR SUPPORT WHEELS ON ROTARY BRE,1.0,68034.00,V200281,OAKWOOD MINING MINERALS
637,A000776,2021-08-13,4510002381,10.0,"REPR;SUPPORT;ROLLER AND SHAFT,ON :",SER:REPAIR SUPPORT WHEELS ON ROTARY BRE,1.0,68034.00,V200281,OAKWOOD MINING MINERALS
1983,A002447,2021-12-07,4510007158,10.0,"TRNSP RD,PROD,HAZ;EXPL&DTNTR,16 CSE",Transportation of Explosives,382.0,2409.46,V200039,MERICA (PTY) LTD
1984,A002448,2021-12-07,4510007158,10.0,"TRNSP RD,PROD,HAZ;EXPL&DTNTR,100-140 CSE",Transportation of Explosives,382.0,3559.36,V200039,MERICA (PTY) LTD
1985,A002449,2021-12-07,4510007158,10.0,"TRNSP RD,PROD,HAZ;EXPL&DTNTR,100-140 CSE",Transportation of Explosives,382.0,3559.36,V200039,MERICA (PTY) LTD
1986,A002450,2021-12-07,4510007158,10.0,"TRNSP RD,PROD,HAZ;EXPL&DTNTR,16 CSE",Transportation of Explosives,382.0,2409.46,V200039,MERICA (PTY) LTD
2065,A002545,2021-12-14,4510007158,10.0,"TRNSP RD,PROD,HAZ;EXPL&DTNTR,16 CSE",Transportation of Explosives,382.0,2409.46,V200039,MERICA (PTY) LTD


## 11. Company B description quality
Descriptions such as blank, `Product`, `Parts` or `No Object` may be weak text signals. Check whether other fields such as material number, GL description or cost centre description can help later.

In [12]:
b_desc = invoices_b["line_description"].astype("string").str.strip()
placeholder_upper = {x.upper() for x in PLACEHOLDERS}

b_description_missing = b_desc.isna() | b_desc.eq("")
b_description_placeholder = b_desc.str.upper().isin(placeholder_upper).fillna(False)

description_quality = pd.DataFrame({
    "metric": ["blank_or_null", "placeholder_or_generic"],
    "records": [int(b_description_missing.sum()),
                int((b_description_placeholder & ~b_description_missing).sum())],
})
display(description_quality)

description_by_doc_type = pd.DataFrame({
    "document_type": invoices_b["document_type"],
    "blank_or_null": b_description_missing,
    "placeholder_or_generic": b_description_placeholder & ~b_description_missing,
}).groupby("document_type", dropna=False).agg(
    total_records=("blank_or_null", "size"),
    blank_or_null=("blank_or_null", "sum"),
    placeholder_or_generic=("placeholder_or_generic", "sum"),
)
description_by_doc_type["blank_pct"] = (
    description_by_doc_type["blank_or_null"] /
    description_by_doc_type["total_records"] * 100
).round(2)
display(description_by_doc_type.sort_values("blank_pct", ascending=False))

display(
    invoices_b.loc[
        b_description_missing | b_description_placeholder,
        ["line_id", "line_description", "material_number",
         "gl_account", "gl_account_description", "vendor_id", "document_type"]
    ].head(30)
)


,metric,records
0,blank_or_null,16695
1,placeholder_or_generic,0


,total_records,blank_or_null,placeholder_or_generic,blank_pct
document_type,,,,
AB,47,47,0,100.0
PD,130,130,0,100.0
PR,58,58,0,100.0
PP,34,34,0,100.0
PM,88,88,0,100.0
Z7,96,96,0,100.0
Z6,169,169,0,100.0
R3,3,3,0,100.0
R1,11,11,0,100.0


,line_id,line_description,material_number,gl_account,gl_account_description,vendor_id,document_type
0,B000001,NaN,MAT-1000007,GL40007,BLCN,V500009,PD
1,B000002,NaN,MAT-1000007,GL40007,BLCN,V500009,PD
2,B000003,NaN,MAT-1000007,GL40007,BLCN,V500009,PD
3,B000004,NaN,MAT-1000007,GL40012,VAPN,V500014,PV
4,B000005,NaN,MAT-1000016,GL40007,VAPN,V500019,PV
5,B000006,NaN,MAT-1000022,GL40007,VAPI,V500023,PV
7,B000008,NaN,MAT-1000007,GL40012,VAPN,V500014,PV
8,B000009,NaN,MAT-1000029,GL40007,VAPI,V500028,PV
9,B000010,NaN,MAT-1000030,GL40007,VAPN,V500029,PV
10,B000011,NaN,MAT-1000038,GL40007,VAPN,V500029,PV


## 12. Company B seed-label coverage and exclusion patterns
Seed labels cover only a subset of B. Treat the labelled subset as imperfect evidence. Unlabelled records are unknown, not automatically excluded.

In [13]:
seed_merge = invoices_b.merge(
    seed_labels[["line_id", "include", "exclusion_reason",
                 "category_l1", "category_l2", "category_l3"]],
    on="line_id",
    how="left",
    validate="one_to_one"
)

labelled_mask = seed_merge["include"].notna()
print(f"Seed-label coverage: {labelled_mask.sum():,} / {len(seed_merge):,} "
      f"({labelled_mask.mean() * 100:.2f}%)")
print("Unlabelled records:", int((~labelled_mask).sum()))

print("Inclusion values in labelled rows:")
display(seed_labels["include"].value_counts(dropna=False).to_frame("records"))

print("Exclusion reasons:")
display(seed_labels["exclusion_reason"].value_counts(dropna=False).to_frame("records"))

doc_label_summary = (
    seed_merge.groupby("document_type", dropna=False)
    .agg(
        total_records=("line_id", "size"),
        labelled_records=("include", "count"),
        included_labelled=("include", lambda s: int((s == 1).sum())),
        excluded_labelled=("include", lambda s: int((s == 0).sum())),
    )
)
doc_label_summary["labelled_inclusion_rate"] = (
    doc_label_summary["included_labelled"] /
    doc_label_summary["labelled_records"].replace(0, np.nan)
)
display(doc_label_summary.sort_values("labelled_records", ascending=False))

excluded_seed = seed_merge.loc[seed_merge["include"] == 0]
display(
    excluded_seed.groupby(["document_type", "exclusion_reason"], dropna=False)
    .size().unstack(fill_value=0)
)

display(
    excluded_seed[
        ["line_id", "document_type", "line_description", "material_number",
         "gl_account_description", "supplier_type", "exclusion_reason"]
    ].head(30)
)


Seed-label coverage: 3,000 / 41,028 (7.31%)
Unlabelled records: 38028
Inclusion values in labelled rows:


,records
include,
1,2550
0,450


Exclusion reasons:


,records
exclusion_reason,
NaN,2550
NON_PROCUREMENT,450


,total_records,labelled_records,included_labelled,excluded_labelled,labelled_inclusion_rate
document_type,,,,,
PV,21138,1785,1540,245,0.862745
RE,18824,1189,991,198,0.833474
PL,77,12,9,3,0.750000
ZG,46,7,6,1,0.857143
ZE,31,3,2,1,0.666667
KG,3,2,2,0,1.000000
PD,130,1,0,1,0.000000
CA,253,1,0,1,0.000000
AB,47,0,0,0,NaN


exclusion_reason,NON_PROCUREMENT
document_type,
CA,1
PD,1
PL,3
PV,245
RE,198
ZE,1
ZG,1


,line_id,document_type,line_description,material_number,gl_account_description,supplier_type,exclusion_reason
2265,B002266,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2350,B002351,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2378,B002379,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2503,B002504,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2505,B002506,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2517,B002518,PV,NaN,MAT-1000007,IC10,NaN,NON_PROCUREMENT
2520,B002521,PV,Sales Tax Expense -,MAT-1000007,PEUS,NaN,NON_PROCUREMENT
2785,B002786,PV,NaN,MAT-1000007,IC37,NaN,NON_PROCUREMENT
2786,B002787,PV,NaN,MAT-1000007,IC37,NaN,NON_PROCUREMENT
2787,B002788,PV,NaN,MAT-1000007,IC37,NaN,NON_PROCUREMENT


## 13. Taxonomy structure and label consistency
Inspect the taxonomy depth and whether the supplied labels belong to the corresponding tree. Do not hardcode category names in the future classifier.

In [14]:
def taxonomy_summary(taxonomy, name):
    level_cols = [c for c in ["category_l1", "category_l2", "category_l3", "category_l4"]
                  if c in taxonomy.columns]
    print(f"===== {name} =====")
    print("Rows:", len(taxonomy))
    print("Columns:", level_cols)
    for col in level_cols:
        print(f"{col}: {taxonomy[col].nunique(dropna=True)} unique values; "
              f"{taxonomy[col].isna().sum()} nulls")
    display(taxonomy.head(10))

taxonomy_summary(taxonomy_a, "Company A taxonomy")
taxonomy_summary(taxonomy_b, "Company B taxonomy")

# Verify Company A golden labels are represented in the taxonomy paths.
a_taxonomy_paths = set(
    taxonomy_a[["category_l1", "category_l2", "category_l3"]]
    .fillna("")
    .astype(str)
    .itertuples(index=False, name=None)
)
a_label_paths = labels_a[["category_l1", "category_l2", "category_l3"]].fillna("").astype(str)
a_label_path_valid = a_label_paths.apply(tuple, axis=1).isin(a_taxonomy_paths)

print("Company A golden-label paths absent from taxonomy:", int((~a_label_path_valid).sum()))

# Company B seed label paths: only assess rows with a nonblank L1.
b_taxonomy_paths = set(
    taxonomy_b[["category_l1", "category_l2", "category_l3"]]
    .fillna("")
    .astype(str)
    .itertuples(index=False, name=None)
)
b_label_rows = seed_labels[["category_l1", "category_l2", "category_l3"]].copy()
b_label_rows = b_label_rows[b_label_rows["category_l1"].notna()]
b_label_paths = b_label_rows.fillna("").astype(str).apply(tuple, axis=1)
print("Company B seed-label paths absent from taxonomy:",
      int((~b_label_paths.isin(b_taxonomy_paths)).sum()))


===== Company A taxonomy =====
Rows: 40
Columns: ['category_l1', 'category_l2', 'category_l3']
category_l1: 5 unique values; 0 nulls
category_l2: 24 unique values; 0 nulls
category_l3: 40 unique values; 0 nulls


,category_l1,category_l2,category_l3
0,Flameproof Diesel Equipment,Flameproof Diesel Equipment Maintenance,Flameproof Diesel Equipment Engines
1,Flameproof Diesel Equipment,Flameproof Diesel Equipment Maintenance,Flameproof Diesel Equipment Starters
2,Flameproof Diesel Equipment,Flameproof Load Haul Dumper,Flameproof Load Haul Dumper
3,Flameproof Diesel Equipment,Flameproof Tractor Trailers,Flameproof Tractor Trailers
4,Flameproof Diesel Equipment,Flameproof Tractors and Multipurpose Vehicles,Flameproof Tractors and Multipurpose Vehicles
5,Mining Operating Materials,Mining Explosives,Mining Explosives
6,Mining Operating Materials,Picks and Sleeves,Picks and Sleeves
7,Mining Operating Materials,Road Maintenance Material,Road Maintenance Material
8,Mining Operating Materials,Roof Bolts,Roof Bolts
9,Mining Operating Materials,Roof Support,Roof Support Accessories


===== Company B taxonomy =====
Rows: 40
Columns: ['category_l1', 'category_l2', 'category_l3']
category_l1: 9 unique values; 0 nulls
category_l2: 25 unique values; 0 nulls
category_l3: 40 unique values; 0 nulls


,category_l1,category_l2,category_l3
0,Direct Materials,Aluminium,Ingot & Metal Premium
1,Direct Materials,Aluminium,Can Body Stock (coil)
2,Direct Materials,Aluminium,Can End Stock (coil)
3,Direct Materials,Aluminium,Tab Stock (coil)
4,Direct Materials,Coatings & Inks,Internal Coatings & Compounds
5,Direct Materials,Coatings & Inks,Inks & Varnishes
6,Direct Materials,Ends & Closures,Can Ends
7,Packaging Materials,Pallets & Load Units,Pallets
8,Packaging Materials,Pallets & Load Units,Top Frames & Layer Pads
9,Packaging Materials,Wrapping & Strapping,"Film, Strapping & Corner Boards"


Company A golden-label paths absent from taxonomy: 0
Company B seed-label paths absent from taxonomy: 0


## 14. Phase 1 findings summary (populate after running)
Use the printed reports above to record observations. Keep observations separate from assumptions. Do not treat correlations (for example, document type versus inclusion) as proof of a business rule.

In [15]:
phase1_checklist = pd.DataFrame([
    ["Data inventory and schemas", "Review Sections 1–2", "Record any unexpected files/columns"],
    ["Nulls, blanks and placeholders", "Review Section 3", "Decide fallback/flag behaviour; don't delete automatically"],
    ["Exact duplicates and IDs", "Review Section 4", "Investigate non-unique IDs before pipeline execution"],
    ["Company B dates", "Review Sections 5–6", "Flag impossible/unparseable dates; don't infer status from paid_date alone"],
    ["Amounts and negatives", "Review Sections 7–9", "Preserve negative values unless evidence justifies exclusion"],
    ["Company A duplicate candidates", "Review Section 10", "Use as candidate pairs only; validate with additional evidence"],
    ["Descriptions", "Review Section 11", "Use other available text/IDs when description is weak"],
    ["B seed labels and exclusions", "Review Section 12", "Labels are imperfect; unlabelled is unknown"],
    ["Taxonomy consistency", "Review Section 13", "Classifier must load taxonomy at runtime"],
], columns=["area", "where_to_review", "decision_prompt"])

display(phase1_checklist)


,area,where_to_review,decision_prompt
0,Data inventory and schemas,Review Sections 1–2,Record any unexpected files/columns
1,"Nulls, blanks and placeholders",Review Section 3,Decide fallback/flag behaviour; don't delete automatically
2,Exact duplicates and IDs,Review Section 4,Investigate non-unique IDs before pipeline execution
3,Company B dates,Review Sections 5–6,Flag impossible/unparseable dates; don't infer status from paid_date alone
4,Amounts and negatives,Review Sections 7–9,Preserve negative values unless evidence justifies exclusion
5,Company A duplicate candidates,Review Section 10,Use as candidate pairs only; validate with additional evidence
6,Descriptions,Review Section 11,Use other available text/IDs when description is weak
7,B seed labels and exclusions,Review Section 12,Labels are imperfect; unlabelled is unknown
8,Taxonomy consistency,Review Section 13,Classifier must load taxonomy at runtime


## Phase 1 exit criteria

Before moving to Phase 2, make sure:

1. All eight datasets load and required columns are present.
2. You understand which fields are missing, blank or generic.
3. Duplicate IDs and exact duplicate rows have been checked.
4. Company B date anomalies have been inspected without assuming every anomaly is an exclusion.
5. Amounts are analysed numerically; negative values are not automatically discarded.
6. Company A duplicate candidates are treated as candidates, not confirmed duplicates.
7. Company B seed-label coverage and exclusion reasons are understood.
8. Taxonomy paths and label consistency have been checked.
9. The evidence and decisions are written in `DECISIONS.md`.

**Next phase:** implement config-driven ingestion and reusable preprocessing in Python modules. Do not put production pipeline logic in this notebook.
